# Weather Agent

A simple weather agent for five U.S. cities. It uses sample weather data and Python tools to show conditions and alerts.

**Note:** Weather data is simulated, not live.


## Setup

Run each cell in order. The weather tools and tests use standard Python. To run the optional AI agent, you also need Google ADK and access to a Gemini model.

```python
%pip install google-adk
```


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any
import math
import os

print("Python standard-library weather simulation loaded.")

## City weather data

Five cities with sample weather conditions.


In [ ]:
@dataclass(frozen=True)
class WeatherFixture:
    city: str
    state: str
    latitude: float
    longitude: float
    condition: str
    temperature_f: int
    wind_mph: int
    precipitation_probability_pct: int


FIXTURES: tuple[WeatherFixture, ...] = (
    WeatherFixture("Miami", "FL", 25.7617, -80.1918, "Sunny", 105, 8, 10),
    WeatherFixture("Denver", "CO", 39.7392, -104.9903, "Heavy snow", 18, 22, 90),
    WeatherFixture("Seattle", "WA", 47.6062, -122.3321, "Light rain", 56, 7, 75),
    WeatherFixture("Phoenix", "AZ", 33.4484, -112.0740, "Sunny", 116, 5, 0),
    WeatherFixture("New York", "NY", 40.7128, -74.0060, "Severe thunderstorms", 76, 38, 95),
)


def _city_key(name: str) -> str:
    """Normalize a user's city spelling for lookup."""
    return " ".join(name.casefold().strip().replace(",", " ").split())


CITY_INDEX: dict[str, WeatherFixture] = {}
for fixture in FIXTURES:
    CITY_INDEX[_city_key(fixture.city)] = fixture
    CITY_INDEX[_city_key(f"{fixture.city}, {fixture.state}")] = fixture


def resolve_city(city: str) -> dict[str, Any]:
    """Resolve one of five supported US cities to its simulated city-center coordinates.

    Args:
        city: City name, optionally followed by its state abbreviation.

    Returns:
        A status and city/state/latitude/longitude, or an informative error.
    """
    fixture = CITY_INDEX.get(_city_key(city))
    if fixture is None:
        return {"status": "error", "message": "Unsupported city; choose Miami, Denver, Seattle, Phoenix, or New York."}
    return {"status": "success", "city": fixture.city, "state": fixture.state,
            "latitude": fixture.latitude, "longitude": fixture.longitude}


print("Supported cities:", ", ".join(f.city for f in FIXTURES))

## Weather tools

Look up weather using latitude and longitude, then create an alert based on the conditions.


In [ ]:
def get_weather_at_coordinates(latitude: float, longitude: float) -> dict[str, Any]:
    """Get simulated weather for one of five configured US city coordinates.

    Args:
        latitude: Decimal latitude returned by the city-lookup tool.
        longitude: Decimal longitude returned by the city-lookup tool.

    Returns:
        A structured simulated observation or an error when coordinates are unsupported.
    """
    if not (math.isfinite(latitude) and math.isfinite(longitude)):
        return {"status": "error", "message": "Coordinates must be finite numbers."}
    for fixture in FIXTURES:
        if abs(latitude - fixture.latitude) < 0.00001 and abs(longitude - fixture.longitude) < 0.00001:
            return {
                "status": "success", "city": fixture.city, "state": fixture.state,
                "latitude": fixture.latitude, "longitude": fixture.longitude,
                "condition": fixture.condition, "temperature_f": fixture.temperature_f,
                "wind_mph": fixture.wind_mph,
                "precipitation_probability_pct": fixture.precipitation_probability_pct,
                "source": "sample_weather_data", "is_live": False,
            }
    return {"status": "error", "message": "No simulated weather is configured for these coordinates."}


def classify_simulated_weather(observation: dict[str, Any]) -> dict[str, str]:
    """Apply deterministic training-only alert rules to an observation.

    Args:
        observation: A successfully retrieved simulated weather result.

    Returns:
        Alert severity, title, and an explanatory message.
    """
    if observation.get("status") != "success":
        return {"severity": "error", "title": "Weather unavailable", "message": "No valid simulated observation."}
    condition = str(observation["condition"]).casefold()
    temp = int(observation["temperature_f"])
    wind = int(observation["wind_mph"])
    if "thunderstorm" in condition or wind >= 35:
        return {"severity": "severe", "title": "Simulated severe-weather alert",
                "message": "Thunderstorms or damaging winds appear in the training scenario."}
    if "snow" in condition:
        return {"severity": "warning", "title": "Simulated winter-weather alert",
                "message": "Snow conditions appear in the training scenario."}
    if temp >= 110:
        return {"severity": "warning", "title": "Simulated extreme-heat warning",
                "message": "The fixture temperature meets the training threshold of 110 F."}
    if temp >= 100:
        return {"severity": "advisory", "title": "Simulated heat advisory",
                "message": "The fixture temperature meets the training threshold of 100 F."}
    return {"severity": "normal", "title": "Routine weather summary",
            "message": "No training alert threshold was reached."}


def weather_summary(city: str) -> str:
    """Provide a transparent sample summary using the two custom weather tools.

    Args:
        city: Supported U.S. city name.

    Returns:
        A concise simulated weather report or an unsupported-city error.
    """
    location = resolve_city(city)
    if location["status"] != "success":
        return location["message"]
    weather = get_weather_at_coordinates(location["latitude"], location["longitude"])
    if weather["status"] != "success":
        return weather["message"]
    alert = classify_simulated_weather(weather)
    return (f"SIMULATED — {weather['city']}, {weather['state']}: "
            f"{weather['condition']}, {weather['temperature_f']} F, "
            f"wind {weather['wind_mph']} mph. "
            f"{alert['title']}: {alert['message']} "
            "Not live weather or an official NWS warning.")

## Weather examples


In [ ]:
for fixture in FIXTURES:
    print(weather_summary(fixture.city))

## Tests

Check that all five cities return the correct results and that invalid locations are handled.


In [ ]:
def run_offline_tests() -> None:
    """Execute deterministic assertions without ADK or external network access."""
    expected = {"Miami": "advisory", "Denver": "warning", "Seattle": "normal",
                "Phoenix": "warning", "New York": "severe"}
    for city, severity in expected.items():
        loc = resolve_city(city)
        assert loc["status"] == "success", city
        obs = get_weather_at_coordinates(loc["latitude"], loc["longitude"])
        assert obs["status"] == "success" and obs["city"] == city, city
        assert obs["is_live"] is False and obs["source"] == "sample_weather_data"
        assert classify_simulated_weather(obs)["severity"] == severity, city
        assert "SIMULATED" in weather_summary(city)
        print(f"PASS — {city}: {severity}")
    assert resolve_city("miami, fl")["city"] == "Miami"
    assert resolve_city("  NEW   YORK  ")["city"] == "New York"
    assert resolve_city("Boston")["status"] == "error"
    assert get_weather_at_coordinates(0.0, 0.0)["status"] == "error"
    assert get_weather_at_coordinates(float("nan"), 0.0)["status"] == "error"
    assert "Unsupported city" in weather_summary("Boston")
    print("PASS — normalization, unsupported city, invalid coordinates, synthetic-data labeling")
    print("ALL OFFLINE TESTS PASSED")


run_offline_tests()

## Google ADK agent

The agent uses the Python functions as tools. If Google ADK is not installed, you can still run the weather examples and tests.


In [ ]:
try:
    from google.adk.agents import LlmAgent
    from google.adk.runners import InMemoryRunner
    from google.genai import types
    ADK_AVAILABLE = True
except ImportError as exc:
    ADK_AVAILABLE = False
    print("ADK not installed; other tests still work:", exc)

if ADK_AVAILABLE:
    root_agent = LlmAgent(
        name="weather_alert_agent",
        model="gemini-2.5-flash",  # Gemini model
        description="Demonstrates five-city simulated weather retrieval with Python custom tools.",
        instruction=(
            "You are a weather agent. "
            "All weather is simulated and MUST be labeled SIMULATED, NOT LIVE. "
            "For a supported city request, call resolve_city first, then "
            "get_weather_at_coordinates with the returned coordinates. "
            "Call weather_summary to obtain the deterministic alert interpretation. "
            "Do not invent observations, forecasts, alerts, locations, or official NWS notices. "
            "For unsupported cities, return the tool's error instead of improvising. "
            "Keep the response concise and include the note that the results are simulated."
        ),
        tools=[resolve_city, get_weather_at_coordinates, weather_summary],
    )
    print("ADK LlmAgent constructed:", root_agent.name)
else:
    root_agent = None

## Try the agent (optional)

If your environment supports Gemini, change `RUN_LIVE_MODEL_TEST` to `True` to ask the agent about each city. Otherwise, leave it `False`.


In [ ]:
RUN_LIVE_MODEL_TEST = False  # Set True only if Gemini access is configured.

async def ask_agent(city: str) -> str:
    """Ask the ADK agent about a city and collect its final response."""
    if not ADK_AVAILABLE or root_agent is None:
        raise RuntimeError("Install google-adk to run the agent.")
    runner = InMemoryRunner(agent=root_agent, app_name="weather_agent")
    user_id = "workshop_user"
    session_id = "session_" + city.lower().replace(" ", "_")
    await runner.session_service.create_session(
        app_name="weather_agent", user_id=user_id, session_id=session_id
    )
    message = types.Content(role="user", parts=[types.Part(text=f"What is the weather in {city}?")])
    replies: list[str] = []
    async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=message):
        if event.is_final_response() and event.content:
            replies.extend(part.text for part in event.content.parts if part.text)
    return "\n".join(replies)

if RUN_LIVE_MODEL_TEST:
    assert ADK_AVAILABLE, "google-adk is not installed"
    for fixture in FIXTURES:
        answer = await ask_agent(fixture.city)
        print(f"{fixture.city}: {answer}\n")
        assert "simulat" in answer.lower(), "Model failed synthetic-data disclosure"
else:
    print("Skipped agent-level inference — credentials and network are not required for offline grading.")

## Summary

This notebook looks up five cities, returns sample weather, creates alerts, and runs automated tests. The ADK agent is optional.

Weather observations and alerts are simulated; they are not official warnings.
